# Sales Data Analysis with Python

In this notebook, I use a small sample sales file to practise a simple analysis workflow: inspect the data, check its quality, calculate revenue, compare products and cities, and look at the monthly totals.

The data is intentionally small, so the results are useful for learning the analysis steps rather than drawing conclusions about a real business.

## 1. Import the libraries

I use Pandas for working with the table and Matplotlib for the charts.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

## 2. Load the sales data

Keep `sample_sales_data.csv` in the same folder as this notebook. The check below gives a clear message if the file is not there.

In [ ]:
data_path = Path("sample_sales_data.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "sample_sales_data.csv was not found. Place it beside this notebook and run again."
    )

raw_df = pd.read_csv(data_path)

print(f"Rows: {raw_df.shape[0]} | Columns: {raw_df.shape[1]}")
display(raw_df.head())

## 3. Understand the columns

Before calculating anything, I check the column names, data types and basic summary. This helps catch unexpected data before using it.

In [ ]:
print("Column names:")
print(raw_df.columns.tolist())

print("\nData types:")
display(raw_df.dtypes.to_frame("data_type"))

print("\nNumeric summary:")
display(raw_df.describe(include="all").transpose())

## 4. Check and prepare the data

The analysis needs product, quantity, price, city and order date. I convert quantity and price to numbers, parse the date, remove exact duplicate rows, and exclude rows where the required fields are missing or where quantity or price is not positive.

Revenue is recalculated from quantity and price so the calculation is consistent.

In [ ]:
required_columns = ["Product", "Quantity", "Price", "City", "Order_Date"]
missing_columns = [column for column in required_columns if column not in raw_df.columns]

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

df = raw_df.copy()

# Clean text fields and convert the numeric/date fields.
for column in ["Product", "City"]:
    df[column] = df[column].astype("string").str.strip()

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

duplicate_count = int(df.duplicated().sum())
missing_before = df[required_columns].isna().sum()

df = df.drop_duplicates()
df = df.dropna(subset=required_columns)
df = df[
    (df["Product"] != "") &
    (df["City"] != "") &
    (df["Quantity"] > 0) &
    (df["Price"] > 0)
].copy()

df["Revenue"] = df["Quantity"] * df["Price"]

print("Exact duplicate rows found:", duplicate_count)
print("\nMissing values before cleaning:")
display(missing_before.to_frame("missing_count"))
print(f"Rows before cleaning: {len(raw_df)}")
print(f"Rows after cleaning:  {len(df)}")

if df.empty:
    raise ValueError("No valid rows remain after cleaning. Please review the source CSV.")

display(df.head())

## 5. Basic metrics

These figures give a quick summary of the cleaned sample. Average order revenue is calculated at the row level because each row represents one sample order record.

In [ ]:
total_revenue = df["Revenue"].sum()
total_units = df["Quantity"].sum()
order_count = len(df)
average_order_revenue = df["Revenue"].mean()

kpi_summary = pd.DataFrame({
    "Metric": ["Total revenue", "Units sold", "Order records", "Average revenue per record"],
    "Value": [total_revenue, total_units, order_count, average_order_revenue]
})

display(kpi_summary)

## 6. Product performance

I group the data by product to compare revenue and units sold. The revenue share helps show how much each product contributes to the sample total.

In [ ]:
product_summary = (
    df.groupby("Product", as_index=False)
      .agg(Revenue=("Revenue", "sum"), Units_Sold=("Quantity", "sum"))
      .sort_values("Revenue", ascending=False)
)
product_summary["Revenue_Share_%"] = (
    product_summary["Revenue"] / total_revenue * 100
).round(1)

display(product_summary)

plt.figure(figsize=(8, 4))
plt.bar(product_summary["Product"], product_summary["Revenue"])
plt.title("Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Revenue (₹)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 7. City comparison

This summary compares the revenue recorded against each city in the sample file. It does not tell us why one city has a higher total; we would need more context and more data to investigate that.

In [ ]:
city_summary = (
    df.groupby("City", as_index=False)
      .agg(Revenue=("Revenue", "sum"), Units_Sold=("Quantity", "sum"))
      .sort_values("Revenue", ascending=False)
)

display(city_summary)

plt.figure(figsize=(8, 4))
plt.bar(city_summary["City"], city_summary["Revenue"])
plt.title("Revenue by City")
plt.xlabel("City")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Monthly revenue

I group orders by calendar month and sort the month values by date. Sorting by the actual month date avoids putting month names in alphabetical order.

In [ ]:
monthly_summary = (
    df.assign(Month=df["Order_Date"].dt.to_period("M").dt.to_timestamp())
      .groupby("Month", as_index=False)
      .agg(Revenue=("Revenue", "sum"), Units_Sold=("Quantity", "sum"))
      .sort_values("Month")
)

display(monthly_summary)

plt.figure(figsize=(8, 4))
plt.plot(monthly_summary["Month"], monthly_summary["Revenue"], marker="o")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (₹)")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Look at the highest-value order record

This identifies the row with the highest calculated revenue in the sample. It is a record-level comparison, not a customer-level analysis.

In [ ]:
highest_order = df.loc[df["Revenue"].idxmax()]

print("Highest-revenue order record:")
display(highest_order.to_frame("Value"))

## 10. Notes from this sample

I print the top product and top city based on the grouped summaries. These are descriptive observations only. The dataset has very few rows, so a larger dataset would be needed before making business recommendations.

In [ ]:
top_product = product_summary.iloc[0]
top_city = city_summary.iloc[0]

print(f"Highest revenue product in this sample: {top_product['Product']} "
      f"(₹{top_product['Revenue']:,.0f})")
print(f"Highest revenue city in this sample: {top_city['City']} "
      f"(₹{top_city['Revenue']:,.0f})")
print(f"Total sample revenue: ₹{total_revenue:,.0f}")
print(f"Number of valid order records: {order_count}")

print("\nLimitations:")
print("- This is a small sample dataset, not a real business dataset.")
print("- There is no customer, discount, cost, profit or returns information.")
print("- The data is not large enough to establish long-term trends.")

## 11. Export the cleaned data

The cleaned file is written next to the notebook. The `.gitignore` file in this repository excludes this generated output so that running the notebook does not create an unnecessary change in Git.

In [ ]:
output_path = Path("cleaned_sales_data.csv")
df.to_csv(output_path, index=False)

print(f"Cleaned data saved to: {output_path.resolve()}")

## Conclusion

This exercise helped me practise loading a CSV file, checking data quality, preparing columns, calculating a measure, grouping records and presenting results with charts.

A useful next step would be to repeat the same workflow with a larger real-world dataset and investigate additional questions such as monthly product mix, average selling price and profit (when cost data is available).